# Building a RAG Pipeline with MongoDB Vector Search

# Connect Google Gemini Modal & Test

In [ ]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
import os

load_dotenv()

llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", api_key=os.getenv("GOOGLE_API_KEY"))

llm.invoke("Hello, how are you?")


AIMessage(content='I am doing well, thank you for asking! How are you today?', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.0-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019af263-29af-73a3-8c5c-9c43f82e29f6-0', usage_metadata={'input_tokens': 6, 'output_tokens': 16, 'total_tokens': 22, 'input_token_details': {'cache_read': 0}})

# Prepare Hugging Face Embeddings Instead of Google Generative AI (Exceeded the quota limit)

In [15]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

def get_embedding(text, input_type="document"):
    """
    Get embeddings for the given text using HuggingFace.
    
    Args:
        text: The text to embed (can be a string or list of strings)
        input_type: Type of input - "document" or "query" (not used for HuggingFace, kept for compatibility)
    
    Returns:
        Embedding vector(s)
    """
    # Note: HuggingFace embeddings don't have task_type like Google embeddings
    # The same model is used for both documents and queries
    
    # Get embeddings
    if isinstance(text, str):
        # Single text
        embedding = embeddings.embed_query(text)
        print(f"Embedding dimension: {len(embedding)}")
        return embedding
    else:
        # Multiple texts
        embedding_list = embeddings.embed_documents(text)
        print(f"Generated {len(embedding_list)} embeddings, dimension: {len(embedding_list[0])}")
        return embedding_list



# Test Hugging Face Embeddings
# Google Generative AI Embeddings exceeded the quota limit and throws error

In [18]:
single_embedding = get_embedding("This is a sample document", input_type="document")
print(f"Single embedding length: {len(single_embedding)}")
 
# Test the function
query_embedding = get_embedding("What is this about?", input_type="query")
print(f"Query embedding length: {len(query_embedding)}")

query_embedding

Embedding dimension: 384
Single embedding length: 384
Embedding dimension: 384
Query embedding length: 384


[-0.0879046618938446,
 0.0994168296456337,
 -0.03311283886432648,
 0.03411264717578888,
 0.04388239234685898,
 -0.012341992929577827,
 0.1836583912372589,
 -0.04412749037146568,
 -0.021410001441836357,
 -0.0843597874045372,
 0.025350268930196762,
 0.08550813049077988,
 0.007213111966848373,
 -0.04016013815999031,
 -0.06895440816879272,
 0.010614571161568165,
 -0.020548369735479355,
 -0.04058380797505379,
 -0.09320506453514099,
 -0.012461571022868156,
 -0.021837370470166206,
 -0.028770897537469864,
 0.014588125981390476,
 0.055740974843502045,
 -0.05139615014195442,
 0.032724447548389435,
 0.005883608479052782,
 0.04049146547913551,
 0.03409069404006004,
 0.05152105912566185,
 -0.0558798685669899,
 0.09460733830928802,
 -0.005960226524621248,
 0.0291544571518898,
 -0.06861642748117447,
 0.00473001366481185,
 0.10685865581035614,
 0.03248545154929161,
 -0.025271713733673096,
 0.0027388499584048986,
 0.050365034490823746,
 -0.11189096421003342,
 -0.04731502756476402,
 0.01368829794228077,

# Data Ingestion using the Langchain and sample documents

In [19]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load the PDF
loader = PyPDFLoader("https://investors.mongodb.com/node/12236/pdf")
data = loader.load()

# Split the data into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=20)
documents = text_splitter.split_documents(data)



In [20]:
docs_to_insert = [{
    "text": doc.page_content,
    "embedding": get_embedding(doc.page_content)
} for doc in documents]

Embedding dimension: 384
Embedding dimension: 384
Embedding dimension: 384
Embedding dimension: 384
Embedding dimension: 384
Embedding dimension: 384
Embedding dimension: 384
Embedding dimension: 384
Embedding dimension: 384
Embedding dimension: 384
Embedding dimension: 384
Embedding dimension: 384
Embedding dimension: 384
Embedding dimension: 384
Embedding dimension: 384
Embedding dimension: 384
Embedding dimension: 384
Embedding dimension: 384
Embedding dimension: 384
Embedding dimension: 384
Embedding dimension: 384
Embedding dimension: 384
Embedding dimension: 384
Embedding dimension: 384
Embedding dimension: 384
Embedding dimension: 384
Embedding dimension: 384
Embedding dimension: 384
Embedding dimension: 384
Embedding dimension: 384
Embedding dimension: 384
Embedding dimension: 384
Embedding dimension: 384
Embedding dimension: 384
Embedding dimension: 384
Embedding dimension: 384
Embedding dimension: 384
Embedding dimension: 384
Embedding dimension: 384
Embedding dimension: 384


In [21]:
docs_to_insert

[{'text': 'MongoDB, Inc. Announces First Quarter Fiscal 2025 Financial Results\nMay 30, 2024\nFirst Quarter Fiscal 2025 Total Revenue of $450.6 million, up 22% Year-over-Year\nContinued Strong Customer Growth with Over 49,200 Customers as of April 30, 2024\nMongoDB Atlas Revenue up 32% Year-over-Year; 70% of Total Q1 Revenue',
  'embedding': [-0.010065988637506962,
   0.009520405903458595,
   -0.00827543344348669,
   0.03640151396393776,
   -0.031777605414390564,
   -0.04345529153943062,
   -0.10090304911136627,
   0.025192653760313988,
   0.011559938080608845,
   0.059784598648548126,
   -0.06596750766038895,
   0.0541246123611927,
   -0.011226307600736618,
   -0.019266784191131592,
   -0.0619245208799839,
   0.01692754216492176,
   0.02016281709074974,
   -0.10663887113332748,
   0.06556175649166107,
   -0.0021224727388471365,
   -0.005065822973847389,
   -0.015644244849681854,
   0.032565515488386154,
   0.007597234100103378,
   0.05290801823139191,
   -0.04959667846560478,
   -0.04

# Connect Mongo DB to insert all Embeddings in Collection

In [ ]:
from pymongo import MongoClient

# Connect to your MongoDB deployment
client = MongoClient("MONGO_DB_URI")
collection =  client["sample_mflix"]["ragpdf"]

# Insert documents into the collection
result = collection.insert_many(docs_to_insert)

# Create Search Index for the better search or query 

In [32]:
from pymongo.operations import SearchIndexModel
import time

# Create your index model, then create the search index
index_name="vector_index_rag"
search_index_model = SearchIndexModel(
  definition = {
    "fields": [
      {
        "type": "vector",
        "numDimensions": 384,
        "path": "embedding",
        "similarity": "cosine"
      }
    ]
  },
  name = index_name,
  type = "vectorSearch"
)
collection.create_search_index(model=search_index_model)

# Wait for initial sync to complete
print("Polling to check if the index is ready. This may take up to a minute.")
predicate=None
if predicate is None:
   predicate = lambda index: index.get("queryable") is True

while True:
   indices = list(collection.list_search_indexes(index_name))
   if len(indices) and predicate(indices[0]):
      break
   time.sleep(5)
print(index_name + " is ready for querying.")

Polling to check if the index is ready. This may take up to a minute.
vector_index_rag is ready for querying.


# Query the vector store

In [35]:
# Define a function to run vector search queries
def get_query_results(query):
  """Gets results from a vector search query."""

  query_embedding = get_embedding(query, input_type="query")
  print(query_embedding)
  pipeline = [
      {
            "$vectorSearch": {
              "index": "vector_index_rag",
              "queryVector": query_embedding,
              "path": "embedding",
              "numCandidates":384,
              "limit": 5
            }
      }, {
            "$project": {
              "_id": 0,
              "text": 1
         }
      }
  ]

  results = collection.aggregate(pipeline)
  print(results)

  array_of_results = []
  for doc in results:
      array_of_results.append(doc)
  return array_of_results

# Test the sample query search result

In [36]:
# Test the function with a sample query
get_query_results("mongodb vector search")

Embedding dimension: 384
[0.01420526672154665, 0.06666281074285507, -0.024411294609308243, 0.0431072823703289, 0.051642924547195435, -0.0009693988831713796, -0.028989898040890694, -0.0341494120657444, 0.04165973514318466, -0.011187446303665638, -0.015423613600432873, -0.0003694662591442466, 0.008392246440052986, -0.03216972574591637, -0.03849019482731819, 0.03774389252066612, -0.01739606447517872, 0.030720725655555725, 0.1015564501285553, -0.010272699408233166, -0.004409311804920435, -0.01573861390352249, 0.020697373896837234, -0.021840987727046013, -0.018759071826934814, 0.004544938448816538, 0.05209888517856598, -0.05199827998876572, 0.007795020472258329, -0.01769644394516945, 0.04458301514387131, 0.04479855298995972, 0.03096979483962059, 0.08816196769475937, -0.12412035465240479, 0.050520289689302444, -0.03901787847280502, -0.013843887485563755, -0.03628423810005188, -0.03500187397003174, 0.01522104162722826, 0.010610019788146019, -0.006780755706131458, -0.006864936091005802, 0.1328

[{'text': 'of MongoDB  8.0—with significant performance improvements such as faster reads and updates, along with significantly\nfaster bulk inserts and time series queries—and the general availability of Atlas Stream Processing to build sophisticated,\nevent-driven applications with real-time data.'},
 {'text': "that allow development teams to address the growing requirements for today's wide variety of modern applications, all in a unified and consistent user\nexperience. MongoDB  has tens of thousands of customers in over 100 countries. The MongoDB  database platform has been downloaded hundreds of"},
 {'text': 'more of our customers. We also see a tremendous opportunity to win more legacy workloads, as AI has now become a catalyst to modernize these\napplications. MongoDB\'s document-based architecture is particularly well-suited for the variety and scale of data required by AI-powered applications.\xa0\nWe are confident MongoDB  will be a substantial beneficiary of this next wave 

# Invoke LLM to query the vector store to get the answer

In [39]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
import os

load_dotenv()

llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", api_key=os.getenv("GOOGLE_API_KEY"))

# Specify search query, retrieve relevant documents, and convert to string
query = "What are MongoDB's latest AI announcements?"
context_docs = get_query_results(query)
context_string = " ".join([doc["text"] for doc in context_docs])

# Construct prompt for the LLM using the retrieved documents as the context
prompt = f"""Use the following pieces of context to answer the question at the end.
    {context_string}
    Question: {query}
"""

completion = llm.invoke(prompt)

print(completion.content)

Embedding dimension: 384
[-0.042797356843948364, -0.03486017882823944, -0.012685264460742474, 0.07189781963825226, 0.09222523123025894, -0.033361054956912994, -0.03393237665295601, 0.008396860212087631, -0.006087033543735743, 0.04252329841256142, -0.06693541258573532, 0.0049794455990195274, -0.0514051616191864, -0.015304290689527988, -0.025196688249707222, 0.06510749459266663, 0.011039470322430134, -0.06799297034740448, 0.018640652298927307, -0.05287860706448555, -0.013876695185899734, -0.015188279561698437, 0.041026823222637177, 0.005066577810794115, 0.017425669357180595, 0.005911497864872217, -0.007305753882974386, -0.06939242035150528, -0.024530747905373573, -0.010916024446487427, -0.05370186269283295, 0.026958804577589035, 0.09249401092529297, -0.01971539482474327, -0.08679540455341339, 0.026142477989196777, 0.028885873034596443, -0.07754278928041458, 0.030689962208271027, -0.03654399886727333, 0.014321082271635532, -0.10925112664699554, -0.008329312317073345, -0.0416150838136673, 